# Methods for minimizing a function

### Dihotomic method

In [1]:
function dihotomic(f, a, b, eps, delta, max_iters = 1000000)
  i = 0
  while i < max_iters && abs(b - a) > eps
    lambda = (a + b) / 2 - delta
    mu = (a + b) / 2 + delta
    if f(lambda) < f(mu)
      b = mu
    else
      a = lambda
    end
    i += 1
  end

  if abs(b - a) < eps
    res = (a + b) / 2
    return a, b, res, f(res)
  else
    error("Maximum iterations reached without convergence.")
  end
end

dihotomic (generic function with 2 methods)

### Binary search on derivative of concave function

In [2]:
# Binary search for point where df/dx = 0
function dx_bs_min(f, a, b, eps, dx_eps, max_iters = 1000000)
  left = a
  right = b
  
  while(right - left) > eps && max_iters > 0
    mid = (left + right) / 2.0
    df = f(mid) - f(mid - dx_eps)
    
    if df < 0
      left = mid
    else
      right = mid
    end

    max_iters -= 1
  end
  
  res = (left + right) / 2.0
  return left, right, res, f(res)
end

dx_bs_min (generic function with 2 methods)

### Golden section method

In [3]:
function golden_section(f, a, b, eps, alpha, max_iters = 1000000)
  lambda = a + (1 - alpha) * (b - a)
  mu = a + alpha * (b - a)
  f_lambda = f(lambda)
  f_mu = f(mu)

  while abs(b - a) > eps && max_iters > 0
    if f_lambda >= f_mu
      a = lambda
      lambda = mu
      f_lambda = f_mu
      mu = a + alpha * (b - a)
      f_mu = f(mu)
    else
      b = mu
      mu = lambda
      f_mu = f_lambda
      lambda = a + (1 - alpha) * (b - a)
      f_lambda = f(lambda)
    end

    max_iters -= 1
  end

  res = (a + b) / 2.0
  return a, b, res, f(res)
end

golden_section (generic function with 2 methods)

### Fibonacci method

In [4]:
function fibonacci(n)
  if n > 70
    return fibonacci_exact(n)
  else
    return fibonacci_approx(n)
  end
end

function fibonacci_exact(n)
  if n <= 0
    return 0
  elseif n == 1
    return 1
  end
  
  a, b = 0, 1
  for _ in 2:n
    a, b = b, a + b
  end
  
  return b
end

function fibonacci_approx(n)
  phi = (1 + sqrt(5)) / 2
  return round(Int, (phi^n - (-phi)^(-n)) / sqrt(5))
end

function fibonacci_search(f, a, b, eps, max_iters = 1000000)
  # Fn = (phi^n - (-phi^(-n))) / sqrt(5) > (b1 - a1) / eps
  phi = (1 + sqrt(5)) / 2
  target = (b - a) / eps
  n = ceil(Int, log(target * sqrt(5)) / log(phi))
  
  n = min(n, max_iters)

  lambda = a + (b - a) * fibonacci(n - 2) / fibonacci(n)
  miu = a + (b - a) * fibonacci(n - 1) / fibonacci(n)
  f_lambda = f(lambda)
  f_miu = f(miu)

  for k in 1:n-2
    if f_lambda >= f_miu
      a = lambda
      lambda = miu
      f_lambda = f_miu
      miu = a + (b - a) * fibonacci(n - k - 1) / fibonacci(n - k)
      f_miu = f(miu)
    else
      b = miu
      miu = lambda
      f_miu = f_lambda
      lambda = a + (b - a) * fibonacci(n - k - 2) / fibonacci(n - k)
      f_lambda = f(lambda)
    end
  end

  miu = lambda + eps
  f_miu = f(miu)

  if f_lambda >= f_miu
    a = lambda
  else
    b = miu
  end

  res = (a + b) / 2.0
  return a, b, res, f(res)
end

fibonacci_search (generic function with 2 methods)

## Biggest decrease method

In [ ]:
using Optim
using ForwardDiff

function biggest_decrease(f, x0, eps, max_iters = 10000)
  x = float.(copy(x0))
  d = zero(x)
  lambda = 0.0
  
  for k in 1:max_iters
    grad_f = ForwardDiff.gradient(f, x)
    
    if norm(grad_f) < eps
      return x, f(x), lambda, d
    end
    
    d = -grad_f
    
    fct = L -> f(x + L * d)
    lambda = Optim.minimizer(optimize(fct, 0.0, 10.0))
    
    x = x + lambda * d
  end
  
  error("Hit maximum iterations without converging.")
  return x, f(x), lambda, d
end

biggest_decrease (generic function with 2 methods)

### Comparison and plotting of all methods

In [15]:
using Plots
using Printf
using BenchmarkTools
using LinearAlgebra

function calculate_min(f, a, b, eps = 1e-4)
  dh_a, dh_b, dh_min, dh_f_min = dihotomic(f, a, b, eps, eps / 10)
  bs_a, bs_b, bs_min, bs_f_min = dx_bs_min(f, a, b, eps, eps / 100)
  gm_a, gm_b, gm_min, gm_f_min = golden_section(f, a, b, eps, (sqrt(5) - 1) / 2)
  fib_a, fib_b, fib_min, fib_f_min = fibonacci_search(f, a, b, eps)
  bd_min, bd_f_min, bd_l, bd_d = biggest_decrease(f, [a], eps)

  @printf("Dihotomic method: ")
  @btime dihotomic($f, $a, $b, $eps, 1e-6)
  @printf("Minimum found in the interval: [%.4f, %.4f]\n", dh_a, dh_b)
  @printf("With x ~= %.4f and f(x) min ~= %.4f\n", dh_min, dh_f_min)
  
  @printf("\nBinary search method: ")
  @btime dx_bs_min($f, $a, $b, $eps, 1e-8)
  @printf("Minimum found in the interval: [%.4f, %.4f]\n", bs_a, bs_b)
  @printf("With x ~= %.4f and f(x) min ~= %.4f\n", bs_min, bs_f_min)

  @printf("\nGolden Section method: ")
  @btime golden_section($f, $a, $b, $eps, (sqrt(5) - 1) / 2)
  @printf("Minimum found in the interval: [%.4f, %.4f]\n", gm_a, gm_b)
  @printf("With x ~= %.4f and f(x) min ~= %.4f\n", gm_min, gm_f_min)

  @printf("\nFibonacci Search method: ")
  @btime fibonacci_search($f, $a, $b, $eps)
  @printf("Minimum found in the interval: [%.4f, %.4f]\n", fib_a, fib_b)
  @printf("With x ~= %.4f and f(x) min ~= %.4f\n", fib_min, fib_f_min)

  @printf("\nBiggest Decrease method: ")
  @btime biggest_decrease($f, [$a], $eps)
  @printf("Minimum found at x ~= %.4f with f(x) min ~= %.4f\n", bd_min[1], bd_f_min)
  @printf("Last step size (lambda): %.4f, last direction norm %.4f\n", bd_l, norm(bd_d))

  # Plot the function and mark the findings
  x = a:0.01:b
  y = f.(x)
  p = plot(x, y, label="f(x)", xlabel="x", ylabel="f(x)")

  annotate!(p, a, maximum(y), text(" Min: $(round(gm_f_min, digits=4)) at x ≈ $(round(gm_min, digits=4))", :left, 10))
  
  return p
end

calculate_min (generic function with 2 methods)

### Results for different functions and intervals

In [23]:
f = x -> x[1] - x[2] + 2*x[1]^2 + x[1]*x[2] + x[2]^2
biggest_decrease(f, 1, 1e-1)

DimensionMismatch: DimensionMismatch: gradient(f, x) expects that x is an array. Perhaps you meant derivative(f, x)?

In [ ]:
functions = [
  (name="f(x) = x[1] - x[2] + 2*x[1]^2+x[1]*x[2]+x[2]^2", f = x -> x[1] - x[2] + 2*x[1]^2 + x[1]*x[2] + x[2]^2, a = 1, b = 1),
  (name="f(x) = x⁴ + x³ + 5x² + x + 2",   f = x -> x[1]^4 + x[1]^3 + 5x[1]^2 + x[1] + 2,     a = -3.0, b = 3.0),
  (name="f(x) = 5x² + x + 16",            f = x -> 5x[1]^2 + x[1] + 16,                a = -4.0, b = 4.0),
  (name="f(x) = sin(x) - cos(x)",         f = x -> sin(x[1]) - cos(x[1]),              a = -2.0, b = 2.0),
  (name="f(x) = eˣ - cos(x) - 5sin(x)",   f = x -> exp(x[1]) - cos(x[1]) - 5sin(x[1]),    a = -7.0, b = -3.0),
  (name="f(x) = x² - x*cos(x)",           f = x -> x[1]^2 - x[1]*cos(x[1]),               a = -3.0, b = 4.0),
  (name="f(x) = 5x³ + 2x² - 1",           f = x -> 5x[1]^3 + 2x[1]^2 - 1,              a = -7.0, b = 3.0),
  (name="f(x) = 5x² + x - 1",             f = x -> 5x[1]^2 + x[1] - 1,                 a = -3.0, b = 3.0),
]

all_plots = []

for fn in functions
  println("\n" * "-"^40)
  println("EVALUATING: $(fn.name)")
  println("-"^40)
  
  p = calculate_min(fn.f, fn.a, fn.b, 1e-1)
  
  title!(p, fn.name) 
  
  push!(all_plots, p)
end

n_cols = 2
n_rows = ceil(Int, length(functions) / n_cols)

plot(all_plots..., layout=(n_rows, n_cols), size=(900, 300 * n_rows))


----------------------------------------
EVALUATING: f(X) = x[1] - x[2] + 2*x[1]^2+x[1]*x[2]+x[2]^2
----------------------------------------


BoundsError: BoundsError: attempt to access Float64 at index [2]